# Phase 2: Representation and Dimensionality Analysis
## Project: Steam "Bridge Titles" Recommender

This notebook implements the **Feature Layer** and **Interaction Layer** representations as specified in the Technical Guide. We apply dimensionality reduction (SVD, PCA) and visualization (UMAP) to identify game clusters and potential "bridge" titles.

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import TruncatedSVD, PCA
import umap
import os

# Set paths
PROCESSED_DIR = '../data/processed'
FIGURES_DIR = '../reports/figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

print("Libraries loaded.")

## 1. Feature Layer Representation (Content-based)

We load the features generated from tags and descriptions (LSA-refined).

In [ ]:
game_features = np.load(os.path.join(PROCESSED_DIR, 'game_features.npz'))['features']
game_ids = pd.read_csv(os.path.join(PROCESSED_DIR, 'game_ids.csv'))
print(f"Game features matrix shape: {game_features.shape}")

## 2. Interaction Layer Representation (Behavior-based)

We load the sparse utility matrix (Users x Games) weighted by hours played.

In [ ]:
interaction_matrix = sp.load_npz(os.path.join(PROCESSED_DIR, 'interaction_matrix.npz'))
app_map = pd.read_csv(os.path.join(PROCESSED_DIR, 'app_map.csv'))
print(f"Interaction matrix shape: {interaction_matrix.shape}")

## 3. Dimensionality Reduction (SVD/PCA)

### 3.1. SVD on Interaction Matrix
This captures latent behavioral patterns. We transpose to get game representations based on user co-occurrence.

In [ ]:
print("Applying SVD to Interaction Matrix (finding latent behavioral themes)...")
svd_inter = TruncatedSVD(n_components=50, random_state=42)
# Transpose because we want embeddings for games (items), matrix is Users x Games
game_behavior_embeddings = svd_inter.fit_transform(interaction_matrix.T)
print(f"Behavioral embeddings shape: {game_behavior_embeddings.shape}")

### 3.2. PCA on Feature Matrix
Further refine the content-based features.

In [ ]:
print("Applying PCA to Feature Matrix...")
pca = PCA(n_components=50)
game_content_embeddings = pca.fit_transform(game_features)
print(f"Content embeddings shape: {game_content_embeddings.shape}")

## 4. Visualization (UMAP)

We visualize the game space to see how genres cluster.

In [ ]:
print("Generating UMAP for Content Layer...")
reducer_content = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
umap_content = reducer_content.fit_transform(game_content_embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(umap_content[:, 0], umap_content[:, 1], s=1, alpha=0.6, cmap='Spectral')
plt.title('UMAP Projection - Content Layer (Tags & Descriptions)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.savefig(os.path.join(FIGURES_DIR, 'umap_content.png'))
plt.show()

In [ ]:
print("Generating UMAP for Interaction Layer...")
reducer_inter = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
umap_inter = reducer_inter.fit_transform(game_behavior_embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(umap_inter[:, 0], umap_inter[:, 1], s=1, alpha=0.6, cmap='coolwarm')
plt.title('UMAP Projection - Interaction Layer (User Co-occurrence)')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.savefig(os.path.join(FIGURES_DIR, 'umap_interaction.png'))
plt.show()

## 5. Conclusion
The visualizations demonstrate that games naturally cluster by both content and behavior. These embeddings will serve as the foundation for identifying **Bridge Titles** in the next phase (Graph Layer).